# 05 · Validation Plan — nitrocefin/carbapenem inhibition (IC50) + controls + β-lactam adjuvant

**Standard slot:** *validation plan.* **For Project 10 this means:** turn the top candidates into a
**costed, controlled wet-lab plan** for the assay that actually tests *inhibition* — an
**enzyme-kinetics INHIBITION assay** (**nitrocefin** chromogenic hydrolysis, or a **carbapenem-
hydrolysis** readout) measuring **IC50** — with the mandatory controls (**off-target
human-metalloenzyme** control, **scrambled-interface** negative, enzyme-only positive), an expression
strategy, and the **β-lactam-adjuvant** stretch (D4/D5).

A design that passes every filter and occludes the channel is a **hypothesis** — **binding ≠
inhibition**; the IC50 assay is what tests it. **No IC50 is generated in silico — that would be
fabricated.** Needs `results/top_candidates.csv` (notebook 04).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the inhibition-assay validation plan

Generate a plan card from the top candidates: the inhibition assay, controls, expression, timeline,
costed reagents. Fill the `<...>` from your own numbers; this is the deliverable other people will
actually read.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if n_top else {}

plan = f"""# NDM-1 Inhibitor (Binder) Validation Plan (Project 10 — by <your name>, <date>)

## Purpose (defensive anti-AMR)
INHIBIT NDM-1 (a carbapenem-hydrolyzing metallo-beta-lactamase) to RESTORE last-resort antibiotic
efficacy. Out of scope: enhancing resistance / pathogen fitness / stabilizing the enzyme.

## Candidates
Top {n_top} candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured. pae_interaction is confidence, occlusion is a
structural proxy -> BINDING != INHIBITION. There is NO in-silico IC50.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (40-80 aa) -> high yield expected.
- NDM-1: express the soluble construct; purify WITH Zn2+ in the buffer to keep the DI-ZINC site intact;
  confirm activity on nitrocefin BEFORE testing binders.

## Assays (go/no-go -> INHIBITION kinetics -> functional)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse binder?).
2. INHIBITION kinetics (the point): pre-incubate NDM-1 with a binder DILUTION SERIES, then add
   substrate and measure residual hydrolysis RATE:
     - nitrocefin (chromogenic cephalosporin; absorbance shift on hydrolysis), or
     - a carbapenem (imipenem/meropenem) hydrolysis readout (UV absorbance drop).
   Fit IC50 (and ideally K_i + mode: competitive / non-competitive / uncompetitive).
3. Stability: DSF (Tm). Deep (optional): co-crystal / cryo-EM of the binder-NDM-1 complex over the di-zinc site.

## Controls (MANDATORY)
- OFF-TARGET human-metalloenzyme control: run the SAME inhibition assay against a human Zn/metalloenzyme
  (e.g. carbonic anhydrase) -> the binder must NOT inhibit it (specificity / safety; mirrors nb-04 specificity).
- Negative (scrambled-interface): YOUR OWN top design with its interface residues scrambled/mutated
  -> must LOSE inhibition (cleanest specificity control).
- Enzyme-only / no-inhibitor positive: NDM-1 + substrate, no binder = 100% activity baseline; a known
  metallo-beta-lactamase chelator/inhibitor probe (e.g. EDTA / a captopril analogue) confirms the assay.

## (Stretch) beta-lactam ADJUVANT readout (the therapeutic point)
Checkerboard of binder x carbapenem (e.g. meropenem) in a resistant strain: does the binder RESTORE
the antibiotic's MIC (synergy / FIC index)? This is the defensive-anti-AMR proof-of-concept: the
binder makes a last-resort antibiotic work again.

## Realistic expectations
In-silico binder hit rates vary widely; the MAJORITY of in-silico hits fail experimentally, and
BINDING != INHIBITION. Expect to test many to find a few real inhibitors. Report the experimental hit
rate (and IC50s) honestly. Do NOT imply a working inhibitor or fabricate an IC50/K_i.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- NDM-1 + human off-target enzyme + nitrocefin/carbapenem substrate + plate reader time: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Defensive anti-AMR: inhibit NDM-1 to restore carbapenem efficacy (in scope, MASTER_BLUEPRINT §7).
Out of scope: enhancing resistance / pathogen fitness. Gene synthesis via a biosecurity-screening
provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its interface residues**
(the positions contacting the NDM-1 rim) — it should **lose** inhibition. Generating these alongside
the real designs (same expression batch) makes the inhibition comparison airtight. Here we scaffold
the sequence-level scramble deterministically; on Colab, scramble the *interface* positions
specifically using the predicted contacts.

In [ ]:
import random
import binder_tools as bt   # bt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted INTERFACE residues specifically (positions contacting the NDM-1 rim)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

import pandas as pd, os
top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
negs = []
if len(top) and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s:
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=bt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control (must LOSE inhibition)"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

## 3 · (Stretch) β-lactam adjuvant + affinity scaffold `[stretch]`

The therapeutic payoff is a **β-lactam adjuvant**: binder + carbapenem restoring the antibiotic's MIC
in a resistant strain (checkerboard / FIC index). And a predicted-affinity tool (Boltz-2) can
**prioritize** which top hits to test first. Use both for **relative ranking + caveats only** —
**never fabricate an IC50, K_i, K_D, or MIC**, and never present a prediction as a measurement.

In [ ]:
# Scaffold ONLY. Do NOT invent IC50/K_i/K_D/MIC. On Colab:
#   - beta-lactam adjuvant: plan a binder x meropenem checkerboard in a blaNDM-1+ resistant strain;
#     report the FIC index / MIC shift from the WET-LAB experiment (not a model).
#   - affinity prioritization: pip install boltz; build the (binder, NDM-1) complex input; run boltz
#     predict; report the RELATIVE ranking of the top hits + heavy caveats (binding != inhibition).
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Stretch scaffold: beta-lactam adjuvant (restore antibiotic MIC) + affinity prioritization.")
print("Relative ranking + caveats only — NEVER a fabricated IC50/K_i/K_D/MIC. Binding != inhibition.")
print("Use it to PRIORITIZE which top hits to test first in the nitrocefin/carbapenem assay — not as evidence.")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **nitrocefin/carbapenem INHIBITION (IC50)** assay, expression, timeline, costed reagents.
- [ ] Controls specified: **off-target human-metalloenzyme** control, **scrambled-interface** negative (`results/negative_controls.csv`), enzyme-only positive.
- [ ] (Stretch) β-lactam-adjuvant checkerboard (MIC restoration) + Boltz-2 affinity for relative ranking — no fabricated IC50/MIC.
- [ ] Honest framing: every design is a hypothesis; **binding ≠ inhibition** until the IC50 assay; report the experimental hit rate.
- [ ] Defensive-anti-AMR framing throughout (inhibit NDM-1, restore carbapenems; never enhance resistance/fitness).
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a defensive anti-AMR binder/inhibitor campaign, honestly reported.